# RAG Bench — 60-Combo Benchmark (Local Jupyter)

한국어 RAG 파이프라인 60개 조합을 로컬 Jupyter 환경에서 벤치마크합니다.

## 3-Layer Architecture
```
Layer 1: Dense Model ─── kosimcse | e5 | bge-m3                   (HuggingFace)
Layer 2: Sparse Model ── korean_bm25 | splade                     (Qdrant built-in)
Layer 3: Reranker ────── flashrank | colbert | none               (Optional)
Layer 4: LLM Support ─── contextual | none                       (Optional, GPT-4o-mini)
```

**2-Pass 실행**:
1. **Pass 1** — 전체 조합 레이턴시 측정 (API 비용 없음)
2. **Pass 2** — 상위 N개 전략만 RAGAS 평가 (LLM API 사용)

체크포인트 지원으로 중단/재개가 가능합니다.

---
## Section 1: 환경 설정

In [ ]:
# Cell 1.2: 의존성 확인
import os
import sys
from pathlib import Path

# ── 프로젝트 경로 추가 ──
_project_root = Path.cwd().parent if Path.cwd().name == "rag_bench_local" else Path.cwd()
for _p in [str(_project_root), str(_project_root / "rag_bench_local")]:
    if _p not in sys.path:
        sys.path.insert(0, _p)

# ── 의존성 확인 ──
try:
    import rag_bench
    import pandas
    import numpy
    print("✅ 핵심 패키지 확인 완료")
except ImportError as e:
    print(f"❌ 패키지 미설치: {e}")
    print("  → pip install -r rag_bench_local/requirements_core.txt")

# ── python-dotenv 확인 ──
try:
    import dotenv
    print("✅ python-dotenv 확인 완료")
except ImportError:
    print("⚠️ python-dotenv 미설치 → pip install python-dotenv")
    print("  (API 키를 .env 파일에서 자동 로드하려면 필요)")

In [ ]:
# Cell 1.3: 로컬 환경 초기화 + rag_bench 패치 + smoke test
import sys
import os
from pathlib import Path

# ── 경로 보장 ──
_project_root = Path.cwd().parent if Path.cwd().name == "rag_bench_local" else Path.cwd()
for _p in [str(_project_root), str(_project_root / "rag_bench_local")]:
    if _p not in sys.path:
        sys.path.insert(0, _p)

# ── 환경 초기화 ──
from rag_bench_local.local_config import init_local
env_info = init_local(qdrant_mode="local")

# ── Smoke Test ──
from rag_bench.strategies import DenseSparseStrategy
print(f"\n✅ rag_bench import 성공: {DenseSparseStrategy.__name__}")

In [4]:
# Cell 1.4: API Key 설정 (자동 로드 실패 시 수동 입력)
import os

if not env_info.get("api_key_loaded", False):
    import getpass
    _key = getpass.getpass("OPENAI_API_KEY 입력 (Enter로 건너뜀): ")
    if _key.strip():
        os.environ["OPENAI_API_KEY"] = _key
        print("✅ OPENAI_API_KEY 설정 완료")
    else:
        print("⚠️ API Key 미설정 — Pass 2 (RAGAS 평가) 실행 불가")
else:
    print("✅ OPENAI_API_KEY 이미 로드됨 (.env 파일)")

In [5]:
# Cell 1.5: Upstage API Key 설정 (자동 로드 실패 시 수동 입력)
import os

if not env_info.get("upstage_api_key_loaded", False):
    import getpass
    _key = getpass.getpass("UPSTAGE_API_KEY 입력 (Enter로 건너뜀): ")
    if _key.strip():
        os.environ["UPSTAGE_API_KEY"] = _key
        print("✅ UPSTAGE_API_KEY 설정 완료")
    else:
        print("ℹ️ Upstage 키 미설정 — Upstage 전략은 사용 불가 (벤치마크 실행에 영향 없음)")
else:
    print("✅ UPSTAGE_API_KEY 이미 로드됨 (.env 파일)")

---
## Section 2: 사용자 설정

### QDRANT_MODE 선택 가이드

| 모드 | 저장 위치 | 세션 종료 후 | 추천 상황 |
|------|-----------|-------------|-----------|
| `local` | `_benchdata/qdrant_db_*` (로컬 파일) | **유지됨** | 일반적인 사용, 재인덱싱 없이 반복 실행 |
| `memory` | 메모리 | **삭제됨** | 빠른 테스트, 일회성 실행 |

In [6]:
# ===== 사용자 설정 =====
PRESET = "full"     # 'quick' (4조합) | 'standard' (24) | 'full' (72)
K = 3                # 검색 결과 수
TOP_N = 6            # Pass 2 RAGAS 평가 대상 (상위 N)

# QDRANT_MODE:
#   'local'    → 로컬 파일 저장 (영속, 재인덱싱 불필요)
#   'memory'   → 인메모리 (빠름, 세션 종료 시 삭제)
QDRANT_MODE = "local"

# RAGAS 메트릭 프리셋:
#   'core_only' (4) → 빠른 평가 (기본값)
#   'comprehensive' (7) → 표준 평가
#   'full' (11+) → 전체 평가
METRIC_PRESET = "core_only"

# 스코어링 프로파일:
#   'balanced' → 4대 메트릭 균등 (기본값)
#   'precision_critical' → 정확도 중심
SCORING_PROFILE = "balanced"

print(f"설정: preset={PRESET}, k={K}, top_n={TOP_N}, qdrant={QDRANT_MODE}")

In [ ]:
# [선택] 로컬 데이터 초기화 — 데이터 오염 방지
# 이전 실행의 캐시가 새 실행에 영향을 줄 수 있을 때 실행하세요.
# 각 플래그를 True로 설정 후 이 셀만 실행합니다.
import glob, shutil, time
from pathlib import Path

# ──────────── 초기화 대상 선택 ─────────────────────
CLEAR_CHECKPOINTS = False   # 체크포인트 초기화
CLEAR_QA = False            # QA 데이터셋 재생성 시
CLEAR_QDRANT = False        # Qdrant 인덱스 재구축 시
# ──────────────────────────────────────────────────

from rag_bench_local.local_config import BENCHDATA_DIR, CHECKPOINTS_DIR

if CLEAR_CHECKPOINTS:
    if CHECKPOINTS_DIR.exists():
        shutil.rmtree(CHECKPOINTS_DIR)
        CHECKPOINTS_DIR.mkdir(parents=True, exist_ok=True)
        print("✅ 체크포인트 초기화 완료")

if CLEAR_QA:
    qa_path = BENCHDATA_DIR / "qa_dataset.json"
    if qa_path.exists():
        qa_path.unlink()
        print("✅ QA 데이터셋 삭제 완료")

if CLEAR_QDRANT:
    for p in BENCHDATA_DIR.glob("qdrant_db_*"):
        if p.is_dir():
            shutil.rmtree(p)
    print("✅ Qdrant 인덱스 초기화 완료")

if not any([CLEAR_CHECKPOINTS, CLEAR_QA, CLEAR_QDRANT]):
    print("ℹ️ 초기화 대상이 없습니다. 플래그를 True로 변경하세요.")

---
## Section 3: QA 데이터셋 생성

PDF 원본에서 RAGAS 기반 QA 쌍을 자동 생성합니다.
이미 `qa_dataset.json`이 존재하면 캐시를 사용하고 건너뜁니다 (`force=True`로 강제 재생성).

> **의존성**: `init_local()`을 먼저 실행해야 경로 패치가 적용됩니다.

In [7]:
# Cell 3.1: 러너 생성
from rag_bench_local.local_runner import LocalBenchmarkRunner

runner = LocalBenchmarkRunner(
    preset=PRESET,
    k=K,
    top_n=TOP_N,
    qdrant_mode=QDRANT_MODE,
    metric_preset=METRIC_PRESET,
    scoring_profile=SCORING_PROFILE,
)
print(f"✅ LocalBenchmarkRunner 생성 (session={runner.session_id})")

In [ ]:
# Cell 3.2: QA 데이터셋 생성 (PDF 페이지 샘플링)
# docs/*.pdf → 10% 샘플링 → data/docs/*.md 재변환 후 RAGAS KG 기반 QA 생성
# QA 수 = 청크 수 × max_qa_per_page (자동 결정)
# 이미 qa_dataset.json이 존재하면 캐시 사용 (force=True로 강제 재생성)
runner.prepare_qa(sample_pages=True)

---
## Section 4: 데이터 로딩

In [ ]:
# Cell 4.1: 데이터 로드 + 청킹
child_chunks, parent_pairs, queries, ground_truths = runner.prepare_data()

print(f"\nQA 샘플:")
for i, q in enumerate(queries[:3]):
    print(f"  Q{i+1}: {q[:80]}...")
    print(f"  A{i+1}: {ground_truths[i][:80]}...")

In [ ]:
# Cell 4.2: Parent-Child 청킹 통계
print(f"Parent 청크: {len(parent_pairs)}개")
print(f"Child 청크: {len(child_chunks)}개")
print(f"\n샘플 Child 청크 (첫 번째):")
print(child_chunks[0].page_content[:300])

---
## Section 5: 조합 생성

In [ ]:
# 프리셋 기반 ComboSpec 생성
combos = runner.generate_combos()

import pandas as pd
combo_table = pd.DataFrame([
    {
        "#": i+1,
        "Label": spec.label,
        "Dense": spec.dense,
        "Sparse": spec.sparse,
        "Reranker": spec.reranker or "-",
        "LLM Support": spec.llm_support or "-",
    }
    for i, spec in enumerate(combos)
])
display(combo_table)

---
## Section 6: Pass 1 — 레이턴시 벤치마크

In [ ]:
# Pass 1: 전체 조합 레이턴시 측정
latency_df = runner.run_pass1(combos, queries, child_chunks, parent_pairs)
display(latency_df)

In [ ]:
# Pass 1 시각화
from rag_bench_local.visualizer import plot_latency_comparison
plot_latency_comparison(latency_df)

---
## Section 7: Pass 2 — RAGAS 평가

In [ ]:
# Pass 2: 상위 N개 전략 RAGAS 평가 (체크포인트 지원)
ragas_df = runner.run_pass2(
    latency_df, combos, queries, ground_truths,
    child_chunks, parent_pairs,
)
display(ragas_df)

In [ ]:
# RAGAS 결과 스타일링 테이블
from rag_bench_local.visualizer import display_styled_table, display_weighted_scores
display_styled_table(ragas_df)

# Weighted Score (프로파일별 가중 점수)
if runner.reports:
    print(f"\n--- Weighted Scores (profile={SCORING_PROFILE}) ---")
    display_weighted_scores(runner.reports, scoring_profile=SCORING_PROFILE)

---
## Section 8: 시각화 대시보드

수행 이력(RunTracker) + RAGAS 차트 + 가중 점수(Weighted Score) 통합 시각화

In [ ]:
# 수행 이력 요약 (RunTracker 연동)
run_record = runner.get_run_record()
if run_record:
    from rag_bench_local.visualizer import plot_run_info, plot_phase_timeline
    print("--- Run Summary ---")
    plot_run_info(run_record)
    print("\n--- Phase Timeline ---")
    plot_phase_timeline(run_record)
else:
    print("RunTracker 데이터가 없습니다.")

In [ ]:
# 히트맵 (전략 x 메트릭)
from rag_bench_local.visualizer import plot_ragas_heatmap
plot_ragas_heatmap(ragas_df)

In [ ]:
# 파레토 프론티어 (레이턴시 vs 품질)
from rag_bench_local.visualizer import plot_latency_vs_quality
plot_latency_vs_quality(latency_df, ragas_df)

In [ ]:
# 레이어별 기여도 분석
from rag_bench_local.visualizer import plot_layer_contribution
plot_layer_contribution(combos, latency_df, metric="avg_latency")

In [ ]:
# [H1] Ablation Waterfall — 레이어별 품질 기여도
from rag_bench_local.visualizer import plot_ablation_waterfall
plot_ablation_waterfall(ragas_df)

In [ ]:
# [H3] Layer Interaction Heatmap — Dense × Sparse 조합 상호작용
from rag_bench_local.visualizer import plot_layer_interaction_heatmap
plot_layer_interaction_heatmap(ragas_df)

In [ ]:
# [H4] Tradeoff Bubble Chart — 레이턴시 × 품질 × 비용
from rag_bench_local.visualizer import plot_tradeoff_bubble
plot_tradeoff_bubble(latency_df, ragas_df, run_record=run_record)

In [ ]:
# [H-2] Metric Violin — 전략 그룹별 per-sample 메트릭 분포
from rag_bench_local.visualizer import plot_metric_violin
if runner.reports:
    plot_metric_violin(runner.reports)
else:
    print("[Info] Pass 2 RAGAS 평가 결과가 필요합니다. (runner.reports 비어있음)")

In [ ]:
# [M-1] Pipeline Diagram — RAG 파이프라인 구조 다이어그램
from rag_bench_local.visualizer import plot_pipeline_diagram
if combos:
    # 대표 조합: contextual+reranker 조합 우선, 없으면 첫 번째
    sample_spec = next(
        (s for s in combos if getattr(s, "reranker", None) and getattr(s, "llm_support", None)),
        combos[0],
    )
    plot_pipeline_diagram(spec=sample_spec)
else:
    print("[Info] combos가 필요합니다. Section 5를 먼저 실행하세요.")

In [ ]:
# [M-3] Strategy Gantt — 전략 빌드 타임라인
from rag_bench_local.visualizer import plot_strategy_gantt
run_record = runner.get_run_record()
if run_record and run_record.get("strategy_timings"):
    plot_strategy_gantt(run_record)
else:
    print("[Info] RunTracker 데이터가 없습니다. Pass 1 실행 후 사용하세요.")

In [ ]:
# [M-4] Cost-Efficiency — 비용 대비 품질 효율 산점도
from rag_bench_local.visualizer import plot_cost_efficiency
run_record = runner.get_run_record()
if run_record and ragas_df is not None and not ragas_df.empty:
    plot_cost_efficiency(run_record, ragas_df, latency_df)
else:
    print("[Info] run_record와 ragas_df가 모두 필요합니다. Pass 1 + Pass 2 실행 후 사용하세요.")

---
## Section 9: 비용 요약 + 결과 저장

In [ ]:
# 비용 요약 (추정)
n_ragas_strategies = len(ragas_df) if ragas_df is not None else 0
n_queries = len(queries)
est_ragas_cost = n_ragas_strategies * n_queries * 0.005  # ~$0.005/query/strategy

cost_data = {
    "RAGAS 평가 (GPT-4o-mini)": est_ragas_cost,
    "Answer 생성 (GPT-4o-mini)": n_ragas_strategies * n_queries * 0.001,
}
total_cost = sum(cost_data.values())

print(f"예상 API 비용:")
for k, v in cost_data.items():
    print(f"  {k}: ${v:.2f}")
print(f"  총계: ${total_cost:.2f}")

from rag_bench_local.visualizer import plot_cost_breakdown
if total_cost > 0:
    plot_cost_breakdown(cost_data)

In [ ]:
# 결과 Export (로컬 파일시스템)
output_dir = runner.export_results(
    latency_df=latency_df,
    ragas_df=ragas_df,
)
print(f"\n결과 저장 위치: {output_dir}")

In [ ]:
# HTML 보고서 열기
from IPython.display import HTML, display
html_path = output_dir / 'report.html'
if html_path.exists():
    display(HTML(f'<a href="{html_path}" target="_blank">📊 HTML 보고서 열기</a>'))
    print(f'HTML 보고서: {html_path}')
else:
    print('[Info] HTML 보고서가 아직 생성되지 않았습니다.')